# Epithelial Cell scANVI Semi-Supervised Annotation

## Overview
This notebook performs scVI pre-training followed by scANVI semi-supervised learning for cell type annotation.

**Input:** `adata_with_manual_annotations.h5ad` (from manual annotation phase)

**Output:** `adata_epithelial_SCANVI_FINAL.h5ad` (with predicted cell types)

**Environment:** scvi-tools environment (separate from bbknn due to conflicts)

## Workflow
1. Data loading and validation
2. Prepare raw counts for scVI
3. scVI pre-training (unsupervised batch correction)
4. scANVI fine-tuning (semi-supervised annotation)
5. Validation and quality control
6. UMAP visualization
7. Final output

---

## Cell 1: Configuration and Environment Setup

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

import os

# Input/Output paths
INPUT_FILE = "/home/h2048/data/py/1206/bbknn_celltype_analysis/Epithelial/annotation_results/adata_with_manual_annotations.h5ad"
OUTPUT_DIR = "/home/h2048/data/py/1206/bbknn_celltype_analysis/Epithelial/scanvi_results"
MODEL_DIR = os.path.join(OUTPUT_DIR, "scanvi_models")

# Create output directories
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# Model hyperparameters
N_LATENT = 30              # Latent space dimensions (10-50)
N_LAYERS = 2               # Number of hidden layers (1-3)
N_EPOCHS_SCVI = 400        # scVI pre-training epochs
N_EPOCHS_SCANVI = 200      # scANVI fine-tuning epochs
BATCH_SIZE = 2048          # Batch size for training
N_HVG = 3000               # Number of highly variable genes

# Annotation columns
BATCH_KEY = 'batch'
MANUAL_LABEL_KEY = 'cell_type_manual'       # From manual annotation
SCANVI_LABEL_KEY = 'cell_type_scanvi_pred'  # scANVI predictions
CLUSTER_KEY = 'leiden_bbknn'

# Random seed for reproducibility
RANDOM_SEED = 42

print("="*80)
print("CONFIGURATION")
print("="*80)
print(f"Input file: {INPUT_FILE}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Model directory: {MODEL_DIR}")
print(f"\nModel parameters:")
print(f"  - Latent dimensions: {N_LATENT}")
print(f"  - Hidden layers: {N_LAYERS}")
print(f"  - scVI epochs: {N_EPOCHS_SCVI}")
print(f"  - scANVI epochs: {N_EPOCHS_SCANVI}")
print(f"  - Batch size: {BATCH_SIZE}")
print(f"  - HVG: {N_HVG}")
print("\n✓ Configuration complete")

## Cell 2: Load Libraries and Check Environment

In [ ]:
# ============================================================================
# LOAD LIBRARIES
# ============================================================================

import warnings
warnings.filterwarnings('ignore')

# Core libraries
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns

# scvi-tools
import scvi
import torch

# Set random seeds
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
scvi.settings.seed = RANDOM_SEED

# Scanpy settings
sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, facecolor='white', frameon=False)

# Check GPU availability
print("="*80)
print("ENVIRONMENT CHECK")
print("="*80)

print(f"\nPython package versions:")
print(f"  - scanpy: {sc.__version__}")
print(f"  - scvi-tools: {scvi.__version__}")
print(f"  - torch: {torch.__version__}")
print(f"  - numpy: {np.__version__}")
print(f"  - pandas: {pd.__version__}")

# GPU check
use_gpu = torch.cuda.is_available()
if use_gpu:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"\n✓ GPU available: {gpu_name}")
    print(f"  Memory: {gpu_memory:.1f} GB")
    scvi.settings.dl_num_workers = 0  # Important for GPU
else:
    print(f"\n⚠️  GPU not available, using CPU")
    print(f"  Training will be slower (~4-6 hours vs ~40 min)")

print(f"\n✓ Libraries loaded successfully")

## Cell 3: Load and Validate Data

In [ ]:
# ============================================================================
# LOAD DATA
# ============================================================================

print("="*80)
print("DATA LOADING AND VALIDATION")
print("="*80)

print(f"\nLoading: {INPUT_FILE}")
adata = sc.read_h5ad(INPUT_FILE)

print(f"\n✓ Data loaded")
print(f"  Shape: {adata.shape[0]:,} cells × {adata.shape[1]:,} genes")

# ============================================================================
# VALIDATE REQUIRED COLUMNS
# ============================================================================

print(f"\nValidating required columns...")

required_cols = [BATCH_KEY, MANUAL_LABEL_KEY, CLUSTER_KEY]
missing_cols = [col for col in required_cols if col not in adata.obs.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

print(f"✓ All required columns present:")
for col in required_cols:
    print(f"  - {col}")

# ============================================================================
# CHECK ANNOTATION STATUS
# ============================================================================

print(f"\nAnnotation status:")

# Count different annotation categories
label_counts = adata.obs[MANUAL_LABEL_KEY].value_counts()
n_unknown = (adata.obs[MANUAL_LABEL_KEY] == 'Unknown').sum()
n_skip = (adata.obs[MANUAL_LABEL_KEY] == 'Skip').sum()
n_labeled = adata.n_obs - n_unknown - n_skip

print(f"  Total cells: {adata.n_obs:,}")
print(f"  Labeled (for training): {n_labeled:,} ({n_labeled/adata.n_obs*100:.1f}%)")
print(f"  Unknown (to predict): {n_unknown:,} ({n_unknown/adata.n_obs*100:.1f}%)")
print(f"  Skip (to exclude): {n_skip:,} ({n_skip/adata.n_obs*100:.1f}%)")

if n_labeled < adata.n_obs * 0.3:
    print(f"\n⚠️  WARNING: Only {n_labeled/adata.n_obs*100:.1f}% cells are labeled")
    print(f"  Recommend: >50% for good scANVI performance")

# Show cell type distribution
print(f"\nLabeled cell type distribution:")
labeled_types = adata.obs[adata.obs[MANUAL_LABEL_KEY] != 'Unknown']
labeled_types = labeled_types[labeled_types[MANUAL_LABEL_KEY] != 'Skip']
type_counts = labeled_types[MANUAL_LABEL_KEY].value_counts()

print(f"\n{'Cell Type':<30} {'Count':<12} {'Percent':<10}")
print("-"*55)
for cell_type, count in type_counts.head(15).items():
    pct = count / len(labeled_types) * 100
    print(f"{cell_type:<30} {count:<12,} {pct:<10.2f}%")

if len(type_counts) > 15:
    print(f"... and {len(type_counts) - 15} more cell types")

# Check batch distribution
print(f"\nBatch distribution:")
batch_counts = adata.obs[BATCH_KEY].value_counts()
print(f"  Number of batches: {len(batch_counts)}")
print(f"  Cells per batch: {batch_counts.min():,} - {batch_counts.max():,}")

print(f"\n✓ Data validation complete")

## Cell 4: Prepare Data for scVI/scANVI

⚠️ **CRITICAL:** scVI requires raw count data (not log-normalized)

In [ ]:
# ============================================================================
# PREPARE RAW COUNTS
# ============================================================================

print("="*80)
print("PREPARING DATA FOR scVI/scANVI")
print("="*80)

print("\n⚠️  CRITICAL: scVI requires raw count data (not log-normalized)")

# Check for raw counts
if adata.raw is not None:
    print("\n✓ Found adata.raw - using raw counts")
    adata_raw = adata.raw.to_adata()
    # Copy metadata
    adata_raw.obs = adata.obs.copy()
    adata_raw.obsm = adata.obsm.copy()
    adata_raw.uns = adata.uns.copy()
    adata = adata_raw
    print(f"  Shape: {adata.shape[0]:,} cells × {adata.shape[1]:,} genes")
elif 'counts' in adata.layers:
    print("\n✓ Found adata.layers['counts'] - using count layer")
    adata.X = adata.layers['counts'].copy()
else:
    print("\n⚠️  WARNING: Cannot find raw counts!")
    print("  Current adata.X might be log-normalized")
    print("  Checking if data looks like counts...")
    
    # Check if data looks like counts
    test_values = adata.X[:1000, :100].toarray() if hasattr(adata.X, 'toarray') else adata.X[:1000, :100]
    max_val = test_values.max()
    has_decimals = np.any(test_values != test_values.astype(int))
    
    if max_val < 20 and has_decimals:
        print(f"  ❌ Data appears to be log-normalized (max={max_val:.2f}, has decimals)")
        print("  ❌ scVI will NOT work properly with normalized data!")
        print("\n  Please reload data before normalization or use adata.raw")
        raise ValueError("Raw counts required but not found")
    else:
        print(f"  ✓ Data looks like counts (max={max_val:.0f})")

# ============================================================================
# BASIC FILTERING
# ============================================================================

print("\nBasic filtering...")

# Filter genes: keep genes detected in at least 3 cells
print(f"  Before filtering: {adata.shape[1]:,} genes")
sc.pp.filter_genes(adata, min_cells=3)
print(f"  After filtering: {adata.shape[1]:,} genes")

# Filter cells: keep cells with at least 200 genes
n_before = adata.n_obs
sc.pp.filter_cells(adata, min_genes=200)
n_after = adata.n_obs
print(f"  Removed {n_before - n_after} low-quality cells")

# ============================================================================
# REMOVE "Skip" CELLS
# ============================================================================

print("\nRemoving 'Skip' cells...")
n_before = adata.n_obs
adata = adata[adata.obs[MANUAL_LABEL_KEY] != 'Skip'].copy()
n_removed = n_before - adata.n_obs
print(f"  Removed {n_removed:,} cells marked as 'Skip'")
print(f"  Remaining: {adata.n_obs:,} cells")

# ============================================================================
# SELECT HIGHLY VARIABLE GENES (batch-aware)
# ============================================================================

print(f"\nSelecting {N_HVG} highly variable genes (batch-aware)...")

sc.pp.highly_variable_genes(
    adata,
    n_top_genes=N_HVG,
    batch_key=BATCH_KEY,
    flavor='seurat_v3',
    subset=True
)

print(f"✓ Selected {adata.shape[1]:,} HVG")

# ============================================================================
# PREPARE LABEL COLUMN FOR scANVI
# ============================================================================

print("\nPreparing label column for scANVI...")

# Create scanvi label column
# - Known labels: keep as is
# - Unknown: mark as "Unknown" for scANVI to predict
adata.obs['cell_type_scanvi'] = adata.obs[MANUAL_LABEL_KEY].copy()

# Count label types
n_labeled = (adata.obs['cell_type_scanvi'] != 'Unknown').sum()
n_unlabeled = (adata.obs['cell_type_scanvi'] == 'Unknown').sum()

print(f"  Labeled cells (for training): {n_labeled:,} ({n_labeled/adata.n_obs*100:.1f}%)")
print(f"  Unlabeled cells (to predict): {n_unlabeled:,} ({n_unlabeled/adata.n_obs*100:.1f}%)")

# Show label distribution
label_dist = adata.obs['cell_type_scanvi'].value_counts()
print(f"\n  Cell types for training:")
for label, count in label_dist.items():
    if label != 'Unknown':
        print(f"    - {label}: {count:,} cells")

# ============================================================================
# SAVE CHECKPOINT
# ============================================================================

checkpoint_file = os.path.join(OUTPUT_DIR, "adata_scanvi_prepared.h5ad")
print(f"\nSaving checkpoint: {checkpoint_file}")
adata.write_h5ad(checkpoint_file)

print(f"\n✓ Data preparation complete")
print(f"  Final shape: {adata.shape[0]:,} cells × {adata.shape[1]:,} genes")
print(f"  Ready for scVI/scANVI training")

## Cell 5: scVI Pre-training (Unsupervised Batch Correction)

⏱️ **Estimated time:**
- GPU: ~10-30 minutes
- CPU: ~2-4 hours

In [ ]:
# ============================================================================
# scVI PRE-TRAINING
# ============================================================================

import time

print("="*80)
print("scVI PRE-TRAINING (Unsupervised Batch Correction)")
print("="*80)

print(f"\nModel configuration:")
print(f"  - Latent dimensions: {N_LATENT}")
print(f"  - Hidden layers: {N_LAYERS}")
print(f"  - Training epochs: {N_EPOCHS_SCVI}")
print(f"  - Batch size: {BATCH_SIZE}")
print(f"  - Device: {'GPU' if use_gpu else 'CPU'}")

print(f"\nEstimated training time:")
if use_gpu:
    print(f"  GPU: ~10-30 minutes")
else:
    print(f"  CPU: ~2-4 hours")

# ============================================================================
# Setup scVI model
# ============================================================================

print(f"\nSetting up scVI model...")

scvi.model.SCVI.setup_anndata(
    adata,
    batch_key=BATCH_KEY,
    layer=None  # Use adata.X (raw counts)
)

vae = scvi.model.SCVI(
    adata,
    n_latent=N_LATENT,
    n_layers=N_LAYERS,
    gene_likelihood='nb'  # Negative binomial for count data
)

print(f"✓ scVI model created")
print(f"  Parameters: {sum(p.numel() for p in vae.module.parameters()):,}")

# ============================================================================
# Train scVI model
# ============================================================================

print(f"\n{'='*80}")
print(f"Starting scVI training...")
print(f"{'='*80}\n")

start_time = time.time()

vae.train(
    max_epochs=N_EPOCHS_SCVI,
    batch_size=BATCH_SIZE,
    early_stopping=True,
    use_gpu=use_gpu
)

elapsed_time = time.time() - start_time
minutes = int(elapsed_time // 60)
seconds = int(elapsed_time % 60)

print(f"\n{'='*80}")
print(f"✓ scVI training complete!")
print(f"  Time: {minutes} min {seconds} sec")
print(f"{'='*80}")

# ============================================================================
# Save scVI model
# ============================================================================

scvi_model_dir = os.path.join(MODEL_DIR, "scvi_model")
print(f"\nSaving scVI model to: {scvi_model_dir}")
vae.save(scvi_model_dir, overwrite=True)
print(f"✓ scVI model saved")

# ============================================================================
# Generate scVI latent representation
# ============================================================================

print(f"\nGenerating scVI latent representation...")
adata.obsm['X_scvi'] = vae.get_latent_representation()
print(f"✓ X_scvi saved to adata.obsm")
print(f"  Shape: {adata.obsm['X_scvi'].shape}")

# Plot training history
print(f"\nPlotting training history...")
fig, ax = plt.subplots(figsize=(10, 6))
train_elbo = vae.history['elbo_train'][1:]
val_elbo = vae.history['elbo_validation'][1:]

ax.plot(train_elbo, label='Training')
ax.plot(val_elbo, label='Validation')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('ELBO Loss', fontsize=12)
ax.set_title('scVI Training History', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '01_scvi_training_history.png'), dpi=300, bbox_inches='tight')
plt.show()
plt.close()

print(f"\n✓ scVI pre-training phase complete!")

## Cell 6: scANVI Fine-tuning (Semi-supervised Learning)

⏱️ **Estimated time:**
- GPU: ~10-20 minutes
- CPU: ~2-3 hours

In [ ]:
# ============================================================================
# scANVI FINE-TUNING
# ============================================================================

print("="*80)
print("scANVI FINE-TUNING (Semi-supervised Learning)")
print("="*80)

print(f"\nModel configuration:")
print(f"  - Base model: scVI (pre-trained)")
print(f"  - Training epochs: {N_EPOCHS_SCANVI}")
print(f"  - Unlabeled category: 'Unknown'")
print(f"  - Device: {'GPU' if use_gpu else 'CPU'}")

print(f"\nEstimated training time:")
if use_gpu:
    print(f"  GPU: ~10-20 minutes")
else:
    print(f"  CPU: ~2-3 hours")

# ============================================================================
# Setup scANVI model from scVI
# ============================================================================

print(f"\nInitializing scANVI from pre-trained scVI...")

lvae = scvi.model.SCANVI.from_scvi_model(
    vae,
    unlabeled_category='Unknown',
    labels_key='cell_type_scanvi'
)

print(f"✓ scANVI model created")
print(f"  Parameters: {sum(p.numel() for p in lvae.module.parameters()):,}")

# ============================================================================
# Train scANVI model
# ============================================================================

print(f"\n{'='*80}")
print(f"Starting scANVI training...")
print(f"{'='*80}\n")

start_time = time.time()

lvae.train(
    max_epochs=N_EPOCHS_SCANVI,
    batch_size=BATCH_SIZE,
    use_gpu=use_gpu
)

elapsed_time = time.time() - start_time
minutes = int(elapsed_time // 60)
seconds = int(elapsed_time % 60)

print(f"\n{'='*80}")
print(f"✓ scANVI training complete!")
print(f"  Time: {minutes} min {seconds} sec")
print(f"{'='*80}")

# ============================================================================
# Save scANVI model
# ============================================================================

scanvi_model_dir = os.path.join(MODEL_DIR, "scanvi_model")
print(f"\nSaving scANVI model to: {scanvi_model_dir}")
lvae.save(scanvi_model_dir, overwrite=True)
print(f"✓ scANVI model saved")

# ============================================================================
# Generate predictions and latent representation
# ============================================================================

print(f"\nGenerating predictions...")

# Get predictions
adata.obs[SCANVI_LABEL_KEY] = lvae.predict()
print(f"✓ Predictions saved to adata.obs['{SCANVI_LABEL_KEY}']")

# Get prediction probabilities (confidence)
predictions_probs = lvae.predict(soft=True)
adata.obs['scanvi_confidence'] = predictions_probs.max(axis=1)
print(f"✓ Confidence scores saved to adata.obs['scanvi_confidence']")

# Get scANVI latent representation
adata.obsm['X_scanvi'] = lvae.get_latent_representation()
print(f"✓ X_scanvi saved to adata.obsm")
print(f"  Shape: {adata.obsm['X_scanvi'].shape}")

# Add label origin info
adata.obs['label_origin'] = [
    'Original' if label != 'Unknown' else 'Predicted'
    for label in adata.obs['cell_type_scanvi']
]

# ============================================================================
# Plot training history
# ============================================================================

print(f"\nPlotting training history...")
fig, ax = plt.subplots(figsize=(10, 6))
train_elbo = lvae.history['elbo_train'][1:]
val_elbo = lvae.history['elbo_validation'][1:]

ax.plot(train_elbo, label='Training')
ax.plot(val_elbo, label='Validation')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('ELBO Loss', fontsize=12)
ax.set_title('scANVI Training History', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '02_scanvi_training_history.png'), dpi=300, bbox_inches='tight')
plt.show()
plt.close()

# ============================================================================
# Save checkpoint
# ============================================================================

checkpoint_file = os.path.join(OUTPUT_DIR, "adata_scanvi_results.h5ad")
print(f"\nSaving checkpoint: {checkpoint_file}")
adata.write_h5ad(checkpoint_file)

print(f"\n✓ scANVI fine-tuning phase complete!")

## Cell 7: Validation and Quality Control

In [ ]:
# ============================================================================
# VALIDATION AND QUALITY CONTROL
# ============================================================================

print("="*80)
print("VALIDATION AND QUALITY CONTROL")
print("="*80)

# ============================================================================
# 1. Overall prediction statistics
# ============================================================================

print(f"\n1. OVERALL STATISTICS")
print("="*60)

n_total = adata.n_obs
n_original = (adata.obs['label_origin'] == 'Original').sum()
n_predicted = (adata.obs['label_origin'] == 'Predicted').sum()

print(f"Total cells: {n_total:,}")
print(f"Originally labeled: {n_original:,} ({n_original/n_total*100:.1f}%)")
print(f"Newly predicted: {n_predicted:,} ({n_predicted/n_total*100:.1f}%)")

# ============================================================================
# 2. Confidence distribution
# ============================================================================

print(f"\n2. PREDICTION CONFIDENCE")
print("="*60)

mean_conf = adata.obs['scanvi_confidence'].mean()
median_conf = adata.obs['scanvi_confidence'].median()

# Categorize confidence
adata.obs['confidence_category'] = pd.cut(
    adata.obs['scanvi_confidence'],
    bins=[0, 0.5, 0.8, 1.0],
    labels=['Low (<0.5)', 'Medium (0.5-0.8)', 'High (>0.8)']
)

conf_dist = adata.obs['confidence_category'].value_counts()

print(f"Mean confidence: {mean_conf:.3f}")
print(f"Median confidence: {median_conf:.3f}")
print(f"\nConfidence distribution:")
for cat, count in conf_dist.items():
    pct = count / n_total * 100
    print(f"  {cat}: {count:,} ({pct:.1f}%)")

# Quality assessment
if mean_conf > 0.7:
    print(f"\n✓ Good: Mean confidence > 0.7")
elif mean_conf > 0.5:
    print(f"\n⚠️  Moderate: Mean confidence 0.5-0.7")
else:
    print(f"\n❌ Low: Mean confidence < 0.5 (may need more labeled data)")

# ============================================================================
# 3. Label agreement (for originally labeled cells)
# ============================================================================

print(f"\n3. LABEL AGREEMENT (Originally Labeled Cells)")
print("="*60)

original_cells = adata.obs[adata.obs['label_origin'] == 'Original']
agreements = (original_cells['cell_type_scanvi'] == original_cells[SCANVI_LABEL_KEY]).sum()
agreement_rate = agreements / len(original_cells) * 100

print(f"Agreement rate: {agreement_rate:.2f}%")
print(f"  Agreements: {agreements:,} / {len(original_cells):,}")

if agreement_rate > 90:
    print(f"\n✓ Excellent: >90% agreement")
elif agreement_rate > 80:
    print(f"\n✓ Good: >80% agreement")
elif agreement_rate > 70:
    print(f"\n⚠️  Moderate: 70-80% agreement (check disagreements)")
else:
    print(f"\n❌ Low: <70% agreement (may indicate labeling issues)")

# Show disagreements
if agreement_rate < 100:
    disagreements = original_cells[original_cells['cell_type_scanvi'] != original_cells[SCANVI_LABEL_KEY]]
    print(f"\nTop disagreements (manual → predicted):")
    disagree_pairs = list(zip(disagreements['cell_type_scanvi'], disagreements[SCANVI_LABEL_KEY]))
    from collections import Counter
    top_disagree = Counter(disagree_pairs).most_common(10)
    for (manual, pred), count in top_disagree:
        print(f"  {manual} → {pred}: {count} cells")

# ============================================================================
# 4. Cell type distribution
# ============================================================================

print(f"\n4. FINAL CELL TYPE DISTRIBUTION")
print("="*60)

type_counts = adata.obs[SCANVI_LABEL_KEY].value_counts()
type_confidence = adata.obs.groupby(SCANVI_LABEL_KEY)['scanvi_confidence'].agg(['mean', 'std'])

print(f"\n{'Cell Type':<30} {'Count':<12} {'Percent':<10} {'Conf (mean±std)':<20}")
print("-"*75)

summary_data = []
for cell_type, count in type_counts.items():
    pct = count / n_total * 100
    mean_conf = type_confidence.loc[cell_type, 'mean']
    std_conf = type_confidence.loc[cell_type, 'std']
    print(f"{cell_type:<30} {count:<12,} {pct:<10.2f}% {mean_conf:.3f}±{std_conf:.3f}")
    
    summary_data.append({
        'Cell_Type': cell_type,
        'Count': count,
        'Percent': pct,
        'Mean_Confidence': mean_conf,
        'Std_Confidence': std_conf
    })

# Save summary
summary_df = pd.DataFrame(summary_data)
summary_df.to_csv(os.path.join(OUTPUT_DIR, 'scanvi_final_summary.csv'), index=False)
print(f"\n✓ Summary saved to: scanvi_final_summary.csv")

# ============================================================================
# 5. Confidence by cell type (for predicted cells)
# ============================================================================

print(f"\n5. PREDICTION CONFIDENCE BY CELL TYPE (Predicted Cells Only)")
print("="*60)

predicted_cells = adata.obs[adata.obs['label_origin'] == 'Predicted']
if len(predicted_cells) > 0:
    pred_conf_by_type = predicted_cells.groupby(SCANVI_LABEL_KEY)['scanvi_confidence'].agg(['count', 'mean', 'std'])
    pred_conf_by_type = pred_conf_by_type.sort_values('mean', ascending=False)
    
    print(f"\n{'Cell Type':<30} {'Count':<10} {'Mean Conf':<12} {'Std Conf':<12}")
    print("-"*65)
    for cell_type, row in pred_conf_by_type.iterrows():
        print(f"{cell_type:<30} {int(row['count']):<10,} {row['mean']:<12.3f} {row['std']:<12.3f}")
    
    # Flag low-confidence predictions
    low_conf_types = pred_conf_by_type[pred_conf_by_type['mean'] < 0.5]
    if len(low_conf_types) > 0:
        print(f"\n⚠️  Cell types with low mean confidence (<0.5):")
        for cell_type in low_conf_types.index:
            print(f"  - {cell_type}")
else:
    print("  No predicted cells (all were originally labeled)")

# ============================================================================
# 6. Create validation plots
# ============================================================================

print(f"\n6. GENERATING VALIDATION PLOTS")
print("="*60)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Confidence histogram
ax = axes[0, 0]
ax.hist(adata.obs['scanvi_confidence'], bins=50, edgecolor='black', alpha=0.7)
ax.axvline(mean_conf, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_conf:.3f}')
ax.set_xlabel('Prediction Confidence', fontsize=12)
ax.set_ylabel('Number of Cells', fontsize=12)
ax.set_title('Prediction Confidence Distribution', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Confidence by cell type
ax = axes[0, 1]
conf_by_type = adata.obs.groupby(SCANVI_LABEL_KEY)['scanvi_confidence'].mean().sort_values()
conf_by_type.plot(kind='barh', ax=ax, color='steelblue')
ax.axvline(0.8, color='green', linestyle='--', alpha=0.5, label='High (>0.8)')
ax.axvline(0.5, color='orange', linestyle='--', alpha=0.5, label='Medium (0.5-0.8)')
ax.set_xlabel('Mean Confidence', fontsize=12)
ax.set_title('Mean Confidence by Cell Type', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, axis='x')

# Plot 3: Confusion matrix (for originally labeled cells)
ax = axes[1, 0]
from sklearn.metrics import confusion_matrix
import seaborn as sns

if len(original_cells) > 0:
    # Get top cell types for readability
    top_types = original_cells['cell_type_scanvi'].value_counts().head(10).index
    subset = original_cells[original_cells['cell_type_scanvi'].isin(top_types)]
    
    cm = confusion_matrix(
        subset['cell_type_scanvi'],
        subset[SCANVI_LABEL_KEY],
        labels=top_types
    )
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=top_types, yticklabels=top_types,
                cbar_kws={'label': 'Count'})
    ax.set_xlabel('scANVI Predicted', fontsize=12)
    ax.set_ylabel('Manual Label', fontsize=12)
    ax.set_title('Confusion Matrix (Top 10 Types)', fontsize=14, fontweight='bold')
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
    plt.setp(ax.get_yticklabels(), rotation=0, fontsize=8)

# Plot 4: Cell count comparison
ax = axes[1, 1]
type_counts_sorted = type_counts.sort_values(ascending=True)
colors = ['green' if t in original_cells['cell_type_scanvi'].values else 'orange' 
          for t in type_counts_sorted.index]
type_counts_sorted.plot(kind='barh', ax=ax, color=colors)
ax.set_xlabel('Number of Cells', fontsize=12)
ax.set_title('Final Cell Type Distribution', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='green', label='In training set'),
    Patch(facecolor='orange', label='Predicted only')
]
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '03_scanvi_validation.png'), dpi=300, bbox_inches='tight')
plt.show()
plt.close()

print(f"\n✓ Validation plots saved: 03_scanvi_validation.png")

print(f"\n{'='*80}")
print(f"✓ VALIDATION COMPLETE")
print(f"{'='*80}")

## Cell 8: UMAP Visualization

In [ ]:
# ============================================================================
# UMAP VISUALIZATION
# ============================================================================

print("="*80)
print("UMAP VISUALIZATION ON scANVI LATENT SPACE")
print("="*80)

# ============================================================================
# Compute UMAP
# ============================================================================

print(f"\nComputing neighborhood graph on X_scanvi...")
sc.pp.neighbors(adata, use_rep='X_scanvi', n_neighbors=15)

print(f"Computing UMAP...")
sc.tl.umap(adata)

print(f"✓ UMAP computed")
print(f"  Coordinates saved to adata.obsm['X_umap']")

# ============================================================================
# Create comprehensive UMAP overview
# ============================================================================

print(f"\nGenerating UMAP overview (7 panels)...")

fig, axes = plt.subplots(3, 3, figsize=(24, 21))
axes = axes.flatten()

# Panel 1: Predicted cell types
sc.pl.umap(
    adata,
    color=SCANVI_LABEL_KEY,
    ax=axes[0],
    show=False,
    title='scANVI Predicted Cell Types',
    frameon=False,
    s=5
)

# Panel 2: Prediction confidence
sc.pl.umap(
    adata,
    color='scanvi_confidence',
    ax=axes[1],
    show=False,
    title='Prediction Confidence',
    frameon=False,
    cmap='viridis',
    vmin=0,
    vmax=1,
    s=5
)

# Panel 3: Batch
sc.pl.umap(
    adata,
    color=BATCH_KEY,
    ax=axes[2],
    show=False,
    title='Batch (Check Mixing)',
    frameon=False,
    s=3
)

# Panel 4: Label origin
sc.pl.umap(
    adata,
    color='label_origin',
    ax=axes[3],
    show=False,
    title='Label Origin',
    frameon=False,
    palette={'Original': 'blue', 'Predicted': 'red'},
    s=5
)

# Panel 5: Tissue (if available)
if 'tissue' in adata.obs.columns:
    sc.pl.umap(
        adata,
        color='tissue',
        ax=axes[4],
        show=False,
        title='Tissue',
        frameon=False,
        s=5
    )
else:
    axes[4].text(0.5, 0.5, 'Tissue info\nnot available', 
                ha='center', va='center', fontsize=14)
    axes[4].axis('off')

# Panel 6: Confidence categories
sc.pl.umap(
    adata,
    color='confidence_category',
    ax=axes[5],
    show=False,
    title='Confidence Categories',
    frameon=False,
    palette={'High (>0.8)': 'green', 'Medium (0.5-0.8)': 'orange', 'Low (<0.5)': 'red'},
    s=5
)

# Panel 7: Original clusters
if CLUSTER_KEY in adata.obs.columns:
    sc.pl.umap(
        adata,
        color=CLUSTER_KEY,
        ax=axes[6],
        show=False,
        title='Original BBKNN Clusters',
        frameon=False,
        legend_loc='on data',
        legend_fontsize=6,
        s=5
    )
else:
    axes[6].axis('off')

# Hide extra panels
axes[7].axis('off')
axes[8].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '04_scanvi_umap_overview.png'), dpi=300, bbox_inches='tight')
plt.show()
plt.close()

print(f"✓ UMAP overview saved: 04_scanvi_umap_overview.png")

# ============================================================================
# Individual cell type UMAPs
# ============================================================================

print(f"\nGenerating individual cell type UMAPs...")

# Get top 12 cell types by abundance
top_types = adata.obs[SCANVI_LABEL_KEY].value_counts().head(12).index.tolist()

fig, axes = plt.subplots(3, 4, figsize=(20, 15))
axes = axes.flatten()

for idx, cell_type in enumerate(top_types):
    # Create binary column
    is_type = (adata.obs[SCANVI_LABEL_KEY] == cell_type).astype(str)
    
    sc.pl.umap(
        adata,
        color=is_type,
        groups=['True'],
        ax=axes[idx],
        show=False,
        title=cell_type,
        frameon=False,
        palette={'True': 'red', 'False': 'lightgray'},
        s=3,
        legend_loc='none'
    )

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '05_scanvi_cell_types_individual.png'), dpi=300, bbox_inches='tight')
plt.show()
plt.close()

print(f"✓ Individual cell type UMAPs saved: 05_scanvi_cell_types_individual.png")

print(f"\n{'='*80}")
print(f"✓ UMAP VISUALIZATION COMPLETE")
print(f"{'='*80}")

## Cell 9: Save Final Results

In [ ]:
# ============================================================================
# SAVE FINAL RESULTS
# ============================================================================

print("="*80)
print("SAVING FINAL RESULTS")
print("="*80)

# Final file
final_file = os.path.join(OUTPUT_DIR, "adata_epithelial_SCANVI_FINAL.h5ad")

print(f"\nSaving final annotated dataset...")
print(f"File: {final_file}")

# Add metadata
adata.uns['scanvi_params'] = {
    'n_latent': N_LATENT,
    'n_layers': N_LAYERS,
    'scvi_epochs': N_EPOCHS_SCVI,
    'scanvi_epochs': N_EPOCHS_SCANVI,
    'batch_size': BATCH_SIZE,
    'n_hvg': N_HVG,
    'random_seed': RANDOM_SEED
}

# Save
adata.write_h5ad(final_file)

print(f"\n✓ Final dataset saved!")
print(f"  Size: {os.path.getsize(final_file) / 1e9:.2f} GB")

# ============================================================================
# Summary of key columns
# ============================================================================

print(f"\n{'='*80}")
print(f"KEY DATA COLUMNS IN FINAL FILE")
print(f"{'='*80}")

print(f"\n📊 adata.obs (cell metadata):")
print(f"  - {MANUAL_LABEL_KEY}: Original manual annotations")
print(f"  - {SCANVI_LABEL_KEY}: Final scANVI predictions (USE THIS!) ⭐")
print(f"  - scanvi_confidence: Prediction confidence (0-1)")
print(f"  - confidence_category: High/Medium/Low")
print(f"  - label_origin: Original vs Predicted")
print(f"  - {BATCH_KEY}: Batch information")
print(f"  - {CLUSTER_KEY}: Original BBKNN clusters")

print(f"\n🧬 adata.obsm (embeddings):")
print(f"  - X_scvi: scVI latent representation")
print(f"  - X_scanvi: scANVI latent representation")
print(f"  - X_umap: UMAP coordinates (from X_scanvi)")

print(f"\n💾 Models saved in: {MODEL_DIR}")
print(f"  - scvi_model/: Pre-trained scVI")
print(f"  - scanvi_model/: Fine-tuned scANVI")

# ============================================================================
# Generate README
# ============================================================================

readme_content = f"""# Epithelial scANVI Annotation Results

## Overview
This directory contains the results of scVI/scANVI semi-supervised annotation.

## Main Output
**adata_epithelial_SCANVI_FINAL.h5ad** - Final annotated dataset
- Use `adata.obs['{SCANVI_LABEL_KEY}']` for cell type annotations
- Use `adata.obs['scanvi_confidence']` for prediction confidence
- Use `adata.obsm['X_scanvi']` for batch-corrected latent space
- Use `adata.obsm['X_umap']` for visualization

## Statistics
- Total cells: {adata.n_obs:,}
- Genes: {adata.n_vars:,}
- Cell types: {len(adata.obs[SCANVI_LABEL_KEY].unique())}
- Mean prediction confidence: {adata.obs['scanvi_confidence'].mean():.3f}

## Files
1. **adata_scanvi_prepared.h5ad** - Data prepared for scVI
2. **adata_scanvi_results.h5ad** - After scANVI training
3. **adata_epithelial_SCANVI_FINAL.h5ad** - Final with UMAP ⭐
4. **scanvi_final_summary.csv** - Cell type statistics
5. **scanvi_models/** - Trained models (can reload for predictions)

## Figures
1. **01_scvi_training_history.png** - scVI training curve
2. **02_scanvi_training_history.png** - scANVI training curve
3. **03_scanvi_validation.png** - Quality control plots
4. **04_scanvi_umap_overview.png** - Comprehensive UMAP (7 panels)
5. **05_scanvi_cell_types_individual.png** - Individual cell types

## Model Parameters
- Latent dimensions: {N_LATENT}
- Hidden layers: {N_LAYERS}
- scVI epochs: {N_EPOCHS_SCVI}
- scANVI epochs: {N_EPOCHS_SCANVI}
- Batch size: {BATCH_SIZE}
- HVG: {N_HVG}

## Next Steps
Use this dataset for:
- Differential expression analysis
- Trajectory inference
- Cell-cell communication
- Publication figures

---
Generated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}
"""

with open(os.path.join(OUTPUT_DIR, 'README.md'), 'w') as f:
    f.write(readme_content)

print(f"\n✓ README.md created")

# ============================================================================
# FINAL SUMMARY
# ============================================================================

print(f"\n{'='*80}")
print(f"🎉 PIPELINE COMPLETE!")
print(f"{'='*80}")

print(f"\n📁 All results saved to:")
print(f"   {OUTPUT_DIR}")

print(f"\n⭐ Main output file:")
print(f"   {final_file}")

print(f"\n📊 Summary statistics:")
print(f"   - Total cells: {adata.n_obs:,}")
print(f"   - Cell types: {len(adata.obs[SCANVI_LABEL_KEY].unique())}")
print(f"   - Mean confidence: {adata.obs['scanvi_confidence'].mean():.3f}")
print(f"   - Label agreement: {agreement_rate:.1f}%")

print(f"\n💡 Use adata.obs['{SCANVI_LABEL_KEY}'] for final cell type annotations!")

print(f"\n✅ Ready for downstream analysis!")
print(f"{'='*80}")